# LegPy tutorial III
### Photon beams

#### Fernando Arqueros, Jaime Rosado, Victor Moya


It is supposed that the user has already worked with the **LegPy tutorial I** and thus he is familiar with the construction of the objects and the MC parameters of LegPy. This tutorial is focused on the simulation of beams of **electrons** or **positrons**. Annihilation of positrons give rise to the emission of photons and therefore it is also advisable to read the **LegPy tutorial II**.  

Next it will be shown, using a number of examples, the procedures to visualize and analyze the results of the simulation of e$^+$/e$^-$ beams. In all these examples n_part is rather small in order to run the cases in a short time, thus leading to large statistical fluctuations. Obviously the user can enlarge n_part to get better statistics.  

### Previous steps

Again, as explained in **tutorial I**, if you have downloaded LegPy in you PC and are runnig this notebook with Jupyter or a similar application, ignore this step. But if you are using **Google Colab**, first you should clone the last version of the LegPy repository each time you inizialite Colab and add the directory where LegPy is located to the Python search path. Also you need an excel package. 

In [ ]:
#Only for Colab users!!
!git clone https://github.com/JaimeRosado/LegPy.git
import sys
sys.path.insert(0,'/content/LegPy')

# Colab users have to install also this package for managing excel files. 
!pip install xlsxwriter
import xlsxwriter

If not yet, the user has to import several modules as explained in **tutorial I**.

In [ ]:
import LegPy as lpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2) Electron/Positron beam - one medium

The algorithm for the transportation is the same for both electrons and positrons but the later annihilate when their kinetic energy is below the threshold (10 keV by default) giving rise to a pair of 511 keV emitted in oposite directions. In fligth annihilation is neglected.  

### 1.1) Display of electron/positron tracks
Annihilation photons are also displayed

In [ ]:
medium = lpy.Medium(name='Al')
geometry = lpy.Geometry(name='orthohedron', x =.2, y = 0.2, z=.2, n_x=30, n_y=30, n_z=100)
spectrum = lpy.Spectrum(name='mono', E = 1.)
beam = lpy.Beam(particle='positron', name='parallel', theta=30.0, phi=20.0, diam=.02, p_in=(-0.03, 0., 0.))
lpy.Plot_beam(medium, geometry, spectrum, beam, n_part=50)
#Alternative:
#result = lpy.MC(medium, geometry, spectrum, beam, n_part=50, tracks=True)

### 1.2) Range and backscattering studies ###

First, check the CSDA range for an appropriate choice of the size of the medium and the `e_length` (`e_K`) parameters   

In [ ]:
E = 1.0 #MeV
spectrum = lpy.Spectrum(name='mono', E=E)
medium = lpy.Medium(name='Al') # NIST
e_data = medium.e_data
CSDA = np.interp(E, e_data.E_ref, e_data.R_ref) # cm
CSDAum = CSDA * 1.e4 # um
print('CSDA = ', round(CSDA, 3), 'cm')

A step length of CSDA/100 (or `e_K` = 0.95) should be enough to get accurate results

In [ ]:
e_length = CSDAum * 0.01 # um (1/100 of CSDA)
#e_K = 0.95

In [ ]:
geometry = lpy.Geometry(name='cylinder', diam=.25, z=.25, n_x=20, n_y=20, n_z=20)
beam = lpy.Beam(particle = 'positron', name='parallel')
result = lpy.MC(medium, geometry, spectrum, beam, n_part=1000, n_zloc=50,
                e_length=e_length)

**Three histograms:**

- Distribution of **range** values $\frac{{\rm d}N}{{\rm d}z}$in units of (cm$^{-1}$). Normalization: $\Sigma \frac{{\rm d}N}{{\rm d}z} {\rm d}z$ = 1
- **Transmission**: $T(z) = \int_z^{z_{max}} \frac{{\rm d}N}{{\rm d}z}{\rm d}z$, fraction of particles reaching depth z.  
- Angular spectrum of **backscattered** electrons/positrons: $\frac{{\rm d}N}{{\rm d}\theta}(\theta)$ in units of rad$^{-1}$. Normalization: $\Sigma \frac{{\rm d}N}{{\rm d}\theta} {\rm d}\theta = 1$ 

- Plot these histograms  

In [ ]:
result.plot_e_hists()

#### 1.2.1) Range ####

Depending on the practical case, two different definitions of electron range R are used:

1) The depth z of the particle at the end of its path. This definition is used for the first histogram of the above figure.

2) The maximum depth reached by the particle. This definition is associated to the transmission curve obtained experimentally when the number of particles traversing layers of several depths z are measured and is used for the second histogram of the above figure.

Differences between both ranges might be non-negligible in cases with strong backscattering.  

- Both definitions of **range** can be computed and stored in a **dataframe** with three columns
    - Depth (cm) along the axis z (cm)
    - Number of particles with range z per unit depth interval (cm$^{-1}$)
    - Fraction of electrons(positrons) escaping the medium.

- Check its shape

In [ ]:
range_df = result.final_z() #(definition 1)
#range_df = result.max_z() #(definition 2)
range_df.head()

- Plot of R distribution and the corresponding integral function of the electron fraction vs z in either definition. Similar to the plots obtained above but you can personalized them.  

In [ ]:
range_df.plot(kind='scatter', x=0, y=1); # 
range_df.plot(kind='scatter', x=0, y=2); # 

- **Extrapolated range:** It is defined as the distance an electron/positron would travel in a material if the nearly linear decreasing part of its transmission curve is extended linearly until it reaches zero. It can be obtained together with the distribution mode and average with

In [ ]:
ext_R, mode, av = lpy.ext_range(range_df)

- This funcion is also incorporated in the result object. The range `definition` should be set to "final" (default) or "max".

In [ ]:
ext_R, mode, av = result.ext_range(definition="max")

#### 1.2.2) Particle Backscattering ####

- The angle distribution of backscattered particles can be computed and stored in a dataframe.

In [ ]:
back = result.backscattering()
back.head()

- The backscattering coefficiente, $b$, is the fraction of backscattered particles. This parameter can be also defined from the backscattered deposit energy as explained below in xx 

In [ ]:
b = back.sum().iloc[1] * result.e_hists.delta_ang
print('b = ', round(100.*b, 2), '%')

- Plot the angular distribution of backscattered particles

In [ ]:
back.plot(kind='scatter', x=0, y=2);

**Positron** beams produce annihilation **photons**.  

- Histograms of photons escaping the medium can be studied with the tools described in **LegPy_tutorial_II**

In [ ]:
result.plot_gamma_out_hists()

### 1.3) Energy deposition and Dose ###

The methods of the object `result` named **Edep** provide the spatial distribution of the energy released by the involved particles (photons, electron/positrons) in units of keV/cm$^3$, that is Edep(x,y,z), Edep(r,z) or Edep(R) for cartesian, cylindrical or spherical voxelization respectively.

For electron beams, neglecting Bremsstralung losses, all the energy is released in the medium by collisions and therefore Edep allows the calculation of the **Dose** since the medium density is known. Positron beams released a fraction of their energy as annihilation photons. Since beam particles are positrons, electron/positron transportation is activated and therefore the spatial distribution of the Dose is accurately evaluated. For more details see 1.4 in **LegPy Tutorial II**.

By default the electron/positron transportation is performed using an `e_length` value appropriate for the voxel size.

#### 1.3.1) Cylindrical voxelization   ####

Only valid for a cylinder.

In [ ]:
medium = lpy.Medium(name='Al')
geometry = lpy.Geometry(name='cylinder', r =.2, z=.2, n_r=20, n_z=20)
spectrum = lpy.Spectrum(name='mono', E=1.)
beam = lpy.Beam(particle='electron', name='parallel', diam=0.02)
#beam = lpy.Beam(particle='positron', name='parallel', diam=0.02)
result = lpy.MC(medium, geometry, spectrum, beam, n_part=10000)

Spatial distribution can be:

- plotted

In [ ]:
result.plot_Edep()

- stored in dataframes

In [ ]:
Edep_df = result.Edep_to_df()
Edep_df.head()

- written in excel file

In [ ]:
result.Edep_to_excel('Edep_cyl')

- Total energy deposited in the medium can be calculated

In [ ]:
Vp = geometry.delta_v # np. array of pixel volumes for all r values (cm^3)
Ed = Edep_df.mul(geometry.delta_v, axis=1).to_numpy().sum() # 
print('total energy deposit =', round(Ed, 2), 'keV')

Note that above results are averaged over axial angle

#### 1.3.2) Cartesian symmetry   ####

Valid for any geometry.

In [ ]:
medium = lpy.Medium(name='Al')
geometry = lpy.Geometry(name='cylinder', r=0.2, z=0.2, n_x=40, n_y=40, n_z=20)
spectrum = lpy.Spectrum(name='mono', E=1.)
beam = lpy.Beam(particle='electron', name='parallel', diam=0.02)
result = lpy.MC(medium, geometry, spectrum, beam, n_part=10000)

Spatial distribution can be:

- plotted

In [ ]:
result.plot_Edep()

- stored in python binary file with extension .npy. Note that this 3D energy distribution cannot be stored in an excel file or a dataframe, but can be saved in this file for later use.

In [ ]:
result.Edep_to_npy('Edep_cart')

- Total energy deposited in the medium can be calculated

In [ ]:
Vp = geometry.delta_v # pixel volume (cm^3)
Ed = result.Edep.sum() * Vp
print('total energy deposit =', round(Ed, 2), 'keV')

- Personalized plots of the spatial distribution of Edep can be done with this tool:  

In [ ]:
z_ind = [0, 4, 9, 13, 16] # index of z layers.
prof_lev = [3.5, 4., 5., 6., 7.] # dosis levels (adjust according the color bar)
result.plot_Edep_layers(axis="z", indexes=z_ind, c_profiles=True, lev=prof_lev)

In [ ]:
x_ind = [5, 10, 18, 25, 32] # index of z layers.
prof_lev = [3.5, 4., 5., 6., 7.] # dosis levels (adjust according the color bar)
result.plot_Edep_layers(axis="x", indexes=x_ind, c_profiles=True, lev=prof_lev)

#### 1.3.1) Spherical voxelization, Edep(R)   ####

Only available if the electron/positron source is located inside the sphere.

In [ ]:
medium = lpy.Medium(name='Pb')
geometry = lpy.Geometry(name='sphere', r = 5.0 , n_r = 500) # Spherical voxelization
beam = lpy.Beam(name = 'isotropic', particle = 'electron')
spectrum = lpy.Spectrum(name='mono', E = 5.0)
result_e = lpy.MC(medium, geometry, spectrum, beam, n_part = 10000)

Spatial distribution can be:

- plotted

In [ ]:
result_e.plot_Edep()

- stored in dataframes

In [ ]:
Edep_df_e = result_e.Edep_to_df()
Edep_df_e.head()

- written in excel file

In [ ]:
result.Edep_to_excel('Edep_sph')

- An interesting result: Dose of positron vs electron beams

We repeat the above simulation for positrons   

In [ ]:
beam = lpy.Beam(name = 'isotropic', particle = 'positron')
result_p = lpy.MC(medium, geometry, spectrum, beam, n_part = 10000)

and show in a log-log plot both Edep(R) 

In [ ]:
Edep_df_p = result_p.Edep_to_df()
depth_p = Edep_df_p.index
Edep_p = Edep_df_p.values

Edep_df_e = result_e.Edep_to_df()
depth_e = Edep_df_e.index
Edep_e = Edep_df_e.values

plt.figure()
plt.scatter(depth_p, Edep_p, color = 'b', label = 'positron ', marker = 'o', s = 15.)
plt.scatter(depth_e, Edep_e, color = 'r', label = 'electron', marker = 'o', s = 15.)

plt.xlabel('depth(cm)', fontsize = 14)
plt.ylabel('Edep (keV/cm$^3$)', fontsize = 14)
plt.title('Radial dose for 5 MeV electrons and positrons in Lead', fontsize = 14)
plt.grid(True, which = 'both')
plt.xscale('log')
plt.yscale('log')

plt.legend(loc='upper right', fontsize = 15)
plt.show()

The energy released by the annihilation photons of positrons is clearly observed at depths larger the e$^+$/e$^-$ range. 

- Total energy deposited in the medium can be calculated

In [ ]:
Vp = geometry.delta_v # np. array of pixel volumes for all r values (cm^3)
Ed_e = Edep_df_e.mul(geometry.delta_v, axis=0).to_numpy().sum() # 
Ed_p = Edep_df_p.mul(geometry.delta_v, axis=0).to_numpy().sum() # 

print('total energy deposit by electrons =', round(Ed_e, 2), 'keV')
print('total energy deposit by positrons =', round(Ed_p, 2), 'keV')

In a 'infinite' sphere electrons deposit all the kinetic energy, but positrons deposit also the energy equivalent to the rest mass of the annihilated paticles.     

### 1.4) Backscattered energy ###

Backscattering can be also measured from the fraction of energy deposited backwards. It can be obtained in a similar way (both cylndrical or cartesian voxelization) just forcing the beam to start its path in some point inside the medium.     

#### 1.4.1) Cylindrical symmetry   ####

In [ ]:
medium = lpy.Medium(name='Al')
geometry = lpy.Geometry(name='cylinder', r =.3, z=.2, n_r=30, n_z=40)
spectrum = lpy.Spectrum(name='mono', E=1.)
z_in = 0.1
beam = lpy.Beam(particle='electron', name='parallel', diam=0.02, p_in=np.array([0., 0., z_in]))
result = lpy.MC(medium, geometry, spectrum, beam, n_part=10000)
result.plot_Edep()

In [ ]:
Edep_df = result.Edep_to_df()
Vp = geometry.delta_v # np. array of pixel volumes for all r values (cm^3)
Ed = Edep_df.mul(Vp, axis=1).to_numpy().sum() # 
print('total energy deposit =', round(Ed, 2), 'keV')

The energy deposited at z < z_in is the backscattered energy.

In [ ]:
Et = Edep_df.mul(Vp, axis=1).to_numpy()
E_back = Et[0:20,:].sum()
E_for = Et[20:40,:].sum()
b_e = E_back/E_for
print('Fraction of backscattered energy =', round(100. * b_e, 2), '%')

#### 2.4.2) Cartesian symmetry   ####

In [ ]:
medium = lpy.Medium(name='Al')
geometry = lpy.Geometry(name='cylinder', r=0.2, z=0.3, n_x=40, n_y=40, n_z=30)
spectrum = lpy.Spectrum(name='mono', E=1.)
z_in = 0.1
beam = lpy.Beam(particle='electron', name='parallel', diam=0.02, p_in=np.array([0., 0., z_in]))
result = lpy.MC(medium, geometry, spectrum, beam, n_part=10000)

In [ ]:
result.plot_Edep()

In [ ]:
Vp = geometry.delta_v # pixel volume (cm^3)
Edep = result.Edep 
Ed = Edep.sum() * Vp
print('total energy deposit =', round(Ed, 2), 'keV')

The energy deposited at z < z_in is the backscattered energy.

In [ ]:
E_back = Edep[:, :, 0:10].sum() * Vp #back Edep
E_for = Edep[:, :, 10:30].sum() * Vp #for Edep
b_e = E_back/E_for
print('Fraction of backscattered energy =', round(100. * b_e, 2), '%')

## 2) Simulations on two-media geometries

The simulation on geometries with two media can be performed in a similar way but you have to:

- Define (construct) both media.
- Give the position of the interface surface in your geometry.

Note that a common voxelization in used in both media. The result can be analyze with the same tools described  above.

Next you can find some examples.

### 2.1) Display of electron/positron tracks
Annihilation photons are also displayed

In [ ]:
Al = lpy.Medium(name='Al')
Au = lpy.Medium(name='Au')
z_int = 0.04
geometry = lpy.Geometry(name='cylinder', r =.05, z=.05, n_r=20, n_z=20, z_int=z_int)
spectrum = lpy.Spectrum(name='mono', E=1.)
beam = lpy.Beam(particle='electron', name='parallel', diam=0.02)
#beam = lpy.Beam(particle='positron', name='parallel', diam=0.02)
result = lpy.MC([Al, Au], geometry, spectrum, beam, n_part=50, tracks=True)

### 2.2) Transmission and backscattering on a two-media cylinder ###

In [ ]:
E = 1.0 #MeV
spectrum = lpy.Spectrum(name='mono', E=E)
Aluminum = lpy.Medium(name='Al')
Gold = lpy.Medium(name='Au')

e_data_Al = Aluminum.e_data
CSDA_Al = np.interp(E, e_data_Al.E_ref, e_data_Al.R_ref) # cm
CSDAum_Al = CSDA_Al * 1.e4 # um

e_data_Au = Gold.e_data
CSDA_Au = np.interp(E, e_data_Au.E_ref, e_data_Au.R_ref) # cm
CSDAum_Au = CSDA_Au * 1.e4 # um

CSDA = min(CSDA_Al , CSDA_Au)

print('min CSDA = ', round(CSDA, 4), 'cm')
e_length = CSDAum_Au * 0.01 # um 1/100 of smallest CSDA (Au)

In [ ]:
geometry = lpy.Geometry(name='cylinder', diam=.25, z=.25, n_x=20, n_y=20, n_z=20, z_int=.10)
beam = lpy.Beam(particle = 'positron', name='parallel')
result = lpy.MC([Aluminum,Gold], geometry, spectrum, beam, n_part=1000, n_zloc=50,
                e_length=e_length)
result.plot_e_hists()

### 2.3) Energy deposition

In [ ]:
Al = lpy.Medium(name='Al')
Au = lpy.Medium(name='Au')
z_int = 0.04
geometry = lpy.Geometry(name='cylinder', r =.05, z=.05, n_r=20, n_z=20, z_int=z_int)
spectrum = lpy.Spectrum(name='mono', E=1.)
beam = lpy.Beam(particle='electron', name='parallel', diam=0.02)
#beam = lpy.Beam(particle='positron', name='parallel', diam=0.02)
result = lpy.MC([Al, Au], geometry, spectrum, beam, n_part=10000)
result.plot_Edep()